In [41]:
import pandas as pd

# Load the final filtered dataset
review_data = pd.read_json('../data/processed/review_data.jsonl', lines=True)
metadata = pd.read_json('../data/processed/metadata.jsonl', lines=True)

print(f"Loaded {len(review_data)} reviews and {len(metadata)} metadata records")
print("Review data columns:", review_data.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())

Loaded 180153 reviews and 55797 metadata records
Review data columns: ['user_id', 'parent_asin', 'rating', 'timestamp', 'verified_purchase', 'helpful_vote', 'text', 'title', 'reviewTime', 'days_since_start']
Metadata columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'categories', 'details', 'parent_asin', 'bought_together']


### Baseline Collaborative Filtering
Use item-user interaction directly without any additional features.


In [42]:
import pandas as pd

# Assume df has ['reviewerID', 'asin', 'overall', 'reviewTime']
ratings_df = review_data[['user_id', 'parent_asin', 'rating', 'reviewTime']].copy()
ratings_df.head()


,user_id,parent_asin,rating,reviewTime
0,AFIQTGZBJCLNJPF53P73GTBP5DZA,B00WSLYQ7C,5.0,963447286000
1,AEZ4DTGST3I7HJQT6YARRUY6ZFFQ,B00005ARK3,5.0,994480282000
2,AFD3P4WZIH2CUZUNOSHCQZYQQSGQ,B00005ARK3,2.0,996721247000
3,AHPJKDPWMTTMDJ2DELCYP5WEXP5Q,B00005NIMJ,3.0,1023322424000
4,AGYZFVPD5MZR5RG5PV4UWFFWEEJQ,B00005ARK3,5.0,1031659951000


In [43]:
# Step 1 - Data Preparation

import torch

# Map users/items to integer IDs.
user2idx = {u: i for i, u in enumerate(ratings_df['user_id'].unique())}
item2idx = {i: j for j, i in enumerate(ratings_df['parent_asin'].unique())}

# Add the encoded columns
ratings_df['user_idx'] = ratings_df['user_id'].map(user2idx)
ratings_df['item_idx'] = ratings_df['parent_asin'].map(item2idx)

users = torch.tensor(ratings_df['user_idx'].values)
items = torch.tensor(ratings_df['item_idx'].values)
ratings = torch.tensor(ratings_df['rating'].values, dtype=torch.float32)


# Train test split
# For each user, keep their last review as test, and earlier ones as train.
#sort by time
ratings_df = ratings_df.sort_values(by=['user_id', 'reviewTime'])

#Split
test_df = ratings_df.groupby('user_id').tail(1)
train_df = ratings_df.drop(test_df.index)

train_users = torch.tensor(train_df['user_idx'].values)
train_items = torch.tensor(train_df['item_idx'].values)
train_ratings = torch.tensor(train_df['rating'].values, dtype=torch.float32)

test_users = torch.tensor(test_df['user_idx'].values)
test_items = torch.tensor(test_df['item_idx'].values)
test_ratings = torch.tensor(test_df['rating'].values, dtype=torch.float32)


print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 149669
Test size: 30484


In [44]:
# import torch

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {device}")

# # Move tensors to GPU
# train_users = train_users.to(device)
# train_items = train_items.to(device)
# train_ratings = train_ratings.to(device)

# test_users = test_users.to(device)
# test_items = test_items.to(device)
# test_ratings = test_ratings.to(device)

In [ ]:
# Training Loop
import sys
sys.path.append('../scripts/Models')  # Add the Models directory to Python path
from SVDModel import SVDModel  # Import the class
import torch.nn as nn
import torch.optim as optim

n_users, n_items = len(user2idx), len(item2idx)
model = SVDModel(n_users, n_items, n_factors=50, ratings_mean=ratings.mean())


criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)






for epoch in range(5):
    model.train()
    optimizer.zero_grad()
    preds = torch.clamp(model(train_users, train_items), 1.0, 5.0)
    loss = criterion(preds, train_ratings)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss {loss.item():.4f}")


Epoch 1, Loss 5.9077
Epoch 2, Loss 5.5389
Epoch 3, Loss 5.2759
Epoch 4, Loss 5.1006
Epoch 5, Loss 4.9911
Epoch 4, Loss 5.1006
Epoch 5, Loss 4.9911


In [46]:
model.eval()
with torch.no_grad():
    test_preds = torch.clamp(model(test_users, test_items), 1.0, 5.0)
    rmse = torch.sqrt(((test_preds - test_ratings) ** 2).mean())
print("Test RMSE:", rmse.item())

Test RMSE: 2.4433722496032715


In [55]:
# --- Negative item sampling and ranking metrics ---
import torch
import random
import numpy as np

# Set a single seed once for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

def precision_at_k(scores: torch.Tensor, pos_idx: int, k: int = 10) -> float:
    """Precision@k: (# relevant in top-k) / k_eff.
    For leave-one-out (one relevant), this returns 1/k_eff if positive in top-k, else 0.
    Uses k_eff = min(k, len(scores)) for robustness when k > number of candidates.
    """
    k_eff = min(k, scores.size(0))
    if k_eff == 0:
        return 0.0
    topk = torch.topk(scores, k_eff).indices
    hit = (topk == pos_idx).any().item()
    return (1.0 / k_eff) if hit else 0.0

def hit_rate_at_k(scores: torch.Tensor, pos_idx: int, k: int = 10) -> float:
    """HitRate@k (aka recall/hit): 1.0 if positive is in top-k, else 0.0."""
    k_eff = min(k, scores.size(0))
    if k_eff == 0:
        return 0.0
    topk = torch.topk(scores, k_eff).indices
    return 1.0 if (topk == pos_idx).any().item() else 0.0

def ndcg_at_k(scores: torch.Tensor, pos_idx: int, k: int = 10) -> float:
    """NDCG@k for single positive: compute DCG contribution and normalize (IDCG=1)."""
    k_eff = min(k, scores.size(0))
    if k_eff == 0:
        return 0.0
    topk = torch.topk(scores, k_eff).indices
    hit_mask = (topk == pos_idx)
    if hit_mask.any().item():
        rank = hit_mask.nonzero(as_tuple=True)[0].item() + 1  # 1-based
        dcg = 1.0 / np.log2(rank + 1)
        return float(dcg)
    return 0.0

def reciprocal_rank(scores: torch.Tensor, pos_idx: int) -> float:
    """Reciprocal Rank: 1 / (rank of positive) in the full candidate list."""
    sorted_indices = torch.argsort(scores, descending=True)
    pos_loc = (sorted_indices == pos_idx).nonzero(as_tuple=True)
    if pos_loc[0].numel() == 0:
        return 0.0
    rank = int(pos_loc[0].item()) + 1
    return 1.0 / rank

def model_score_for_candidates(model, user_id: int, candidate_items: list, device=None) -> torch.Tensor:
    """Vectorized scoring call matching model(user_tensor, item_tensor) API.
    Returns a 1-D torch tensor (len(candidate_items)) on CPU.
    """
    model.eval()
    with torch.no_grad():
        user_tensor = torch.tensor([user_id] * len(candidate_items), dtype=torch.long, device=device)
        item_tensor = torch.tensor(candidate_items, dtype=torch.long, device=device)
        scores = model(user_tensor, item_tensor)
        scores = scores.view(-1)
        # Keep same clamping as used elsewhere to keep score ranges consistent
        scores = torch.clamp(scores, 1.0, 5.0)
        return scores.cpu()

def evaluate_leave_one_out(model, val_users, pos_item_for_user, all_item_ids,
                           num_negatives: int = 1000, ks=(5, 10, 20), device=None):
    """Leave-one-out ranking evaluation.

    Returns dict with Precision@K, NDCG@K, HitRate@K, MRR and Recall@10.
    """
    metrics_acc = {
        'precision': {k: [] for k in ks},
        'ndcg': {k: [] for k in ks},
        'hit_rate': {k: [] for k in ks},
        'rr': [],
        'recall10': []
    }

    all_items_set = set(all_item_ids)

    for i, user in enumerate(val_users):
        pos = pos_item_for_user[user]
        candidate_pool = list(all_items_set - {pos})
        if len(candidate_pool) <= num_negatives:
            negatives = candidate_pool
        else:
            negatives = random.sample(candidate_pool, num_negatives)

        candidates = negatives + [pos]
        scores = model_score_for_candidates(model, user, candidates, device)
        pos_idx = len(candidates) - 1

        for k in ks:
            p = precision_at_k(scores, pos_idx, k=k)
            nd = ndcg_at_k(scores, pos_idx, k=k)
            hit = hit_rate_at_k(scores, pos_idx, k=k)
            metrics_acc['precision'][k].append(p)
            metrics_acc['ndcg'][k].append(nd)
            metrics_acc['hit_rate'][k].append(hit)

        rr = reciprocal_rank(scores, pos_idx)
        metrics_acc['rr'].append(rr)

    results = {}
    for k in ks:
        results[f'Precision@{k}'] = float(np.mean(metrics_acc['precision'][k])) if metrics_acc['precision'][k] else 0.0
        results[f'NDCG@{k}'] = float(np.mean(metrics_acc['ndcg'][k])) if metrics_acc['ndcg'][k] else 0.0
        results[f'HitRate@{k}'] = float(np.mean(metrics_acc['hit_rate'][k])) if metrics_acc['hit_rate'][k] else 0.0

    results['MRR'] = float(np.mean(metrics_acc['rr'])) if metrics_acc['rr'] else 0.0

    return results

# Example usage (replace with your actual variables):
# val_users = list(test_df['user_idx'].values)
# pos_item_for_user = dict(zip(test_df['user_idx'].values, test_df['item_idx'].values))
# all_item_ids = list(range(n_items))
# device = next(model.parameters()).device
# results = evaluate_leave_one_out(model, val_users, pos_item_for_user, all_item_ids, num_negatives=100, ks=(5,10,20), device=device)
# print(results)

In [49]:
# Pretty-print results dict as a table (run after `results` is available)
import pandas as pd
from collections import defaultdict

# results = {...}  # already present from your eval cell

# Split metrics with '@' (e.g. "Precision@5") into metric x K table
table = defaultdict(dict)
others = {}
for name, val in results.items():
    if "@" in name:
        metric, k = name.split("@", 1)
        try:
            k = int(k)
        except ValueError:
            k = k
        table[metric.strip()][k] = val
    else:
        others[name] = val

# Build DataFrame for @-metrics
ks = sorted({k for metrics in table.values() for k in metrics})
metrics = sorted(table.keys())
df = pd.DataFrame(index=metrics, columns=ks, dtype=float)
for m in metrics:
    for k in ks:
        df.at[m, k] = table[m].get(k, float("nan"))

# Format for display
df_display = df.applymap(lambda x: f"{x:.4f}" if pd.notnull(x) else "")
display(df)        # shows numeric table (good for sorting/inspection)
print("\nFormatted (rounded) table:")
print(df_display.to_markdown())

# Print single-value metrics (MRR, Recall@10, etc.)
if others:
    print("\nOther metrics:")
    for name, val in others.items():
        print(f"- {name}: {val:.4f}")

C:\Users\rahul\AppData\Local\Temp\ipykernel_25004\2767343318.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_display = df.applymap(lambda x: f"{x:.4f}" if pd.notnull(x) else "")


,5,10,20
HitRate,0.035920,0.082404,0.215917
NDCG,0.018723,0.033668,0.067826
Precision,0.007184,0.008240,0.010796



Formatted (rounded) table:
|           |      5 |     10 |     20 |
|:----------|-------:|-------:|-------:|
| HitRate   | 0.0359 | 0.0824 | 0.2159 |
| NDCG      | 0.0187 | 0.0337 | 0.0678 |
| Precision | 0.0072 | 0.0082 | 0.0108 |

Other metrics:
- MRR: 0.0426


In [ ]:
def train_and_evaluate_svd(weight_decay=0.0, epochs=5, lr=0.01, num_negatives=100, ks=(5,10,20), l2_reg_embeddings_only=False, l2_lambda=1e-4, verbose=True):

    import sys
    sys.path.append('../scripts/Models')
    from SVDModel import SVDModel
    import torch.nn as nn
    import torch.optim as optim
    import pandas as pd
    from collections import defaultdict
    
    # Create fresh model
    model = SVDModel(n_users, n_items, n_factors=50, ratings_mean=ratings.mean())
    
    # Setup training
    criterion = nn.MSELoss()
    # Use weight_decay only if not doing custom embeddings-only regularization
    optimizer_weight_decay = 0.0 if l2_reg_embeddings_only else weight_decay
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=optimizer_weight_decay)
    
    if verbose:
        if l2_reg_embeddings_only:
            reg_info = f" (L2 embeddings only: {l2_lambda})"
        elif weight_decay > 0:
            reg_info = f" (L2 reg: {weight_decay})"
        else:
            reg_info = ""
        print(f"Training SVD model for {epochs} epochs{reg_info}")
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = torch.clamp(model(train_users, train_items), 1.0, 5.0)
        loss = criterion(preds, train_ratings)
        if l2_reg_embeddings_only:
            l2_reg = model.user_emb.weight.pow(2).sum() + model.item_emb.weight.pow(2).sum()
            loss = loss + l2_lambda * l2_reg
        loss.backward()
        optimizer.step()
        if verbose:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")
    
    # RMSE evaluation
    model.eval()
    with torch.no_grad():
        test_preds = torch.clamp(model(test_users, test_items), 1.0, 5.0)
        rmse = torch.sqrt(((test_preds - test_ratings) ** 2).mean())
    
    if verbose:
        print(f"Test RMSE: {rmse.item():.4f}")
    
    # Ranking evaluation
    if verbose:
        print("Running leave-one-out evaluation...")
    
    val_users = list(test_df['user_idx'].values)
    pos_item_for_user = dict(zip(test_df['user_idx'].values, test_df['item_idx'].values))
    all_item_ids = list(range(n_items))
    device = next(model.parameters()).device
    
    results = evaluate_leave_one_out(model, val_users, pos_item_for_user, all_item_ids, 
                                   num_negatives=num_negatives, ks=ks, device=device)
    
    # Add RMSE to results
    results['RMSE'] = rmse.item()
    
    # Pretty print results
    if verbose:
        print("\n" + "="*50)
        if l2_reg_embeddings_only:
            print(f"RESULTS (L2 embeddings only: {l2_lambda})")
        else:
            print(f"RESULTS (weight_decay={weight_decay})")
        print("="*50)
        
        # Split metrics with '@' into table format
        table = defaultdict(dict)
        others = {}
        for name, val in results.items():
            if "@" in name:
                metric, k = name.split("@", 1)
                try:
                    k = int(k)
                except ValueError:
                    k = k
                table[metric.strip()][k] = val
            else:
                others[name] = val
        
        # Build DataFrame for @-metrics
        if table:
            ks_found = sorted({k for metrics in table.values() for k in metrics})
            metrics = sorted(table.keys())
            df = pd.DataFrame(index=metrics, columns=ks_found, dtype=float)
            for m in metrics:
                for k in ks_found:
                    df.at[m, k] = table[m].get(k, float("nan"))
            
            print("\nRanking Metrics:")
            print(df.round(4).to_string())
        
        # Print single-value metrics
        if others:
            print(f"\nOther Metrics:")
            for name, val in others.items():
                print(f"  {name}: {val:.4f}")
    
    return results

# Example usage:
print("Training baseline model (no regularization):")
baseline_results = train_and_evaluate_svd(weight_decay=0.0)

print("\n" + "="*60)
print("Training with L2 regularization:")
reg_results = train_and_evaluate_svd(weight_decay=1e-4)

print("\n" + "="*60)
print("Training with L2 regularization on Embeddings:")
reg_results = train_and_evaluate_svd(l2_reg_embeddings_only=True)


Training baseline model (no regularization):
Training SVD model for 5 epochs
Epoch 1/5, Loss: 5.9077
Epoch 2/5, Loss: 5.5389
Epoch 3/5, Loss: 5.2759
Epoch 4/5, Loss: 5.1006
Epoch 5/5, Loss: 4.9911
Test RMSE: 2.4434
Running leave-one-out evaluation...

RESULTS (weight_decay=0.0)

Ranking Metrics:
               5       10      20
HitRate    0.0386  0.0863  0.2175
NDCG       0.0202  0.0355  0.0690
Precision  0.0077  0.0086  0.0109

Other Metrics:
  MRR: 0.0435
  RMSE: 2.4434

Training with L2 regularization:
Training SVD model for 5 epochs (L2 reg: 0.0001)
Epoch 1/5, Loss: 5.8653
Epoch 2/5, Loss: 5.6782
Epoch 3/5, Loss: 5.5177
Epoch 4/5, Loss: 5.3797
Epoch 5/5, Loss: 5.2607
Test RMSE: 2.4077
Running leave-one-out evaluation...

RESULTS (weight_decay=0.0001)

Ranking Metrics:
               5       10      20
HitRate    0.0403  0.0877  0.2215
NDCG       0.0207  0.0359  0.0701
Precision  0.0081  0.0088  0.0111

Other Metrics:
  MRR: 0.0439
  RMSE: 2.4077

Training with L2 regularization on